<img src=../figures/Brown_logo.svg width=50%>

## Data-Driven Design & Analyses of Structures & Materials (3dasm)

## Lecture 19.1

### Elvis Aguero | <a href = "mailto: elvis_alexander_aguero_vera@brown.edu">elvis_alexander_aguero_vera@brown.edu</a>  | PhD candidate

## Introduction

**What:** A lecture of the "3dasm" course, introducing **agentic** data-driven design and
analysis of structures and materials (**adda**).

**Where:** This notebook comes from this [repository](https://github.com/bessagroup/3dasm_course)

**Reference for entire course:** Murphy, Kevin P. *Probabilistic machine learning: an
introduction*. MIT press, 2022. Available online [here](https://probml.github.io/pml-book/book1.html)

**How:** This is the first of two lectures. Today is about **how you build** such a framework
and what we learned doing it. Lecture 19.2 is about **what happened** when we pointed it at a
real research problem.

Speaker notes.

This pair of lectures is the sequel to the f3dasm lecture. f3dasm gives you the blocks of the
data-driven process and asks you to choose and wire them. adda is about what happens when the
*choosing* is itself automated.

Framing for the room: nobody should leave this lecture thinking the machine does the science.
They should leave thinking the machine does an enormous amount of the *work*, and that the
interesting engineering is in making it hard for the machine (and for us) to fool ourselves.

## Outline for today

* Where we are: the data-driven process, and who makes the decisions
* What makes *design* different from most agentic-AI settings
* Six design decisions, and the reason for each
    - the graph and the open loop
    - delegation as the unit of work **and** of accounting
    - a metered oracle and a canonical, append-only ledger
    - falsification instead of a reward
    - an adversarial critic and a reproduction gate
    - guards chosen by reversibility
* Lessons learned (the part that cost us the most)

**Reading material**: this notebook + the a3dasm documentation
([concepts](https://elvis-aguero.github.io/a3dasm/)).

## Recap: the data-driven process

<img src=../figures/f3dasm_overview.svg width=100%>

In `f3dasm` the process is made of blocks you can name:

* **Design** — the parametrization and its bounds
* **Data generation** — the simulator or experiment that scores a design
* **Machine learning** — the surrogate you fit to what you measured
* **Optimization** — how you use the surrogate to decide what to try next

The framework standardizes the blocks. **You** still supply every decision:

*which* parametrization, *which* sampler, *which* surrogate, *which* optimizer,
when to stop, and — hardest of all — **whether the answer you got is real.**

## The question behind today's lecture

We have spent this course learning to make those decisions well.

<br>

> Which of them are *decisions*, and which are *labor*?

<br>

Fitting a GP to 200 points is labor. Choosing to fit a GP at all is a decision.
Writing an Abaqus input deck is labor. Deciding that a new *shape* is worth a week of
compute is a decision.

**adda** is a bet on a specific answer: an LLM-driven system can carry the labor *and* a
useful share of the decisions — **provided** we build the scaffolding that keeps it honest.

That proviso is the whole lecture.

Speaker notes.

Resist the temptation to present this as "we automated science". The honest claim is narrower
and more interesting: a lot of what a graduate student does in a data-driven design study is
mechanical, and a surprising amount of what looks like judgement is actually pattern-matching
against literature and prior runs. Those parts transfer. What does not transfer easily is
knowing when you are being fooled — which is why most of the engineering effort went there.

## What makes *design* different

Agentic frameworks have made real progress on scientific computing tasks. A good recent
example from our own building is **GRAFT-ATHENA** (Toscano, Chai & Karniadakis, 2026), which
assembles and improves numerical methods across physics problems, and its predecessor
ATHENA (Toscano, Chen & Karniadakis, 2025).

<br>

Our problem has three features that shaped every decision we made:

**1. There is no reference solution.**

Nobody knows the best supercompressible mast. There is no analytical answer, no published
optimum, no held-out test set. We cannot compute an error, so we cannot use error as a signal.

**2. Evaluating one design is expensive, and sometimes fails.**

One Abaqus post-buckling solve is minutes to hours, needs a license from a shared pool, and
may not converge. Some evaluations come back *partial*.

**3. The objective has a novelty clause.**

We asked for a design that is *new in shape or arrangement* — not a resized cross-section.
So a number can be **real, reproducible, and still not count.**

## Each feature forces a decision

| Because... | ...we needed |
| :-- | :-- |
| there is no reference solution | a way to test claims that does not need one → **falsification** |
| evaluations are expensive and flaky | a **metered oracle**, budgets, and partial-result handling |
| evaluations are the scarce resource | **provenance on every single one** |
| a real number may not count | a **human-owned contract** for what counts, and a critic that enforces it |
| runs last hours and nobody is watching | **guards**, a watchdog, and an auditable record |

<br>

This is the through-line: **the architecture follows from the epistemic situation**, not from a
preference for one agent topology over another.

Speaker notes.

This table is the spine of the lecture. If a student remembers only one slide, this is the one:
architecture as a consequence of what you can and cannot measure.

Note the deliberate framing — we are describing the constraints *we* face. We are not claiming
other frameworks face fewer or that they got anything wrong; they solve different problems and
their choices follow from their own constraints, exactly as ours do from ours.

# Decision 1
## A graph with a hub, not a pipeline

## The graph

A fixed pipeline — sample, fit, optimize, report — cannot react. Research is not a pipeline;
what you do next depends on what just happened.

So: **one hub and four specialists.**

In [ ]:
# A schematic of the agent graph. Nothing here calls a model - it just draws the topology
# we settled on, so we can talk about it.

import matplotlib.pyplot as plt
import numpy as np

%config InlineBackend.figure_format = "retina"
plt.rcParams["figure.figsize"] = (8, 4)

In [ ]:
def draw_graph():
    """Hub-and-spoke: the strategizer delegates, everyone reports back the same way."""
    fig, ax = plt.subplots(figsize=(9, 4.2))

    hub = (0.5, 0.82)
    specialists = {
        "literature\nreviewer": (0.10, 0.34),
        "data\ngenerator": (0.37, 0.34),
        "implementer": (0.63, 0.34),
        "critic": (0.90, 0.34),
    }

    for name, pos in specialists.items():
        # two-way arrows: delegate down, report back up
        ax.annotate("", xy=pos, xytext=hub,
                    arrowprops=dict(arrowstyle="<->", color="0.45", lw=1.4,
                                    shrinkA=26, shrinkB=26))
        ax.text(*pos, name, ha="center", va="center", fontsize=10,
                bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="0.6"))

    ax.text(*hub, "STRATEGIZER\n(reads, decides, delegates)", ha="center", va="center",
            fontsize=11, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.5", fc="#fdf3d8", ec="0.4"))

    # the open loop: the hub feeds itself
    ax.annotate("", xy=(0.63, 0.95), xytext=(0.37, 0.95),
                arrowprops=dict(arrowstyle="->", color="#b8860b", lw=1.6,
                                connectionstyle="arc3,rad=-0.9"))
    ax.text(0.5, 1.02, "open loop: decide again after every report",
            ha="center", fontsize=9, color="#b8860b", style="italic")

    ax.set_xlim(-0.02, 1.02); ax.set_ylim(0.15, 1.10); ax.axis("off")
    plt.tight_layout(); plt.show()

draw_graph()

## Why a hub, and why "open"

* **literature reviewer** — finds and reads relevant papers
* **data generator** — turns a way of evaluating a design into a metered oracle
* **implementer** — writes and runs the actual code: sampling, surrogates, optimization
* **critic** — an adversarial reviewer that tries to find holes *before* a result is accepted

**Open loop** means there is no script. After every report the strategizer looks at the current
state and chooses the next move. The loop ends when it declares the work done **and that
decision survives review.**

A principle we had to learn to hold: **all nodes are equal.**

Node-specific special-casing — "the critic gets this extra field", "the implementer skips that
check" — is a *defect to remove*, not design intent. Every exception you carve becomes a place
where the system behaves differently than you reason about it.

Speaker notes.

The "all nodes are equal" rule sounds like fussiness. It is not. Every special case is a second
mental model you must hold while debugging, and the whole difficulty of these systems is that
you cannot step through them. Uniformity is what makes them reasonable about at all.

# Decision 2
## Delegation is the unit of work *and* of accounting

## Delegation

When the strategizer hands work to a specialist, that is a **delegation**: an id (`D001`,
`D002`, ...), a task description, and a report that comes back.

<br>

The important half is the second one. Every real evaluation is **attributed to the delegation
that produced it**.

This is what lets the system answer, months later and without trust:

> *Where did this number come from?*

In [ ]:
import pandas as pd

# A toy stand-in for the canonical evaluation ledger. Every row is one REAL solve.
# The leading-underscore columns are provenance stamped by the framework, not by the agent.
ledger = pd.DataFrame({
    "ratio_d":        [0.02005, 0.0092, 0.0188, 0.0120, 0.0151],
    "ratio_pitch":    [0.25,    0.602,  0.480,  0.618,  0.700],
    "sigma_crit":     [0.1306,  0.3644, 0.2210, 0.7111, 0.0904],
    "mcs":            [0.999,   1.00,   0.84,   1.00,   0.31],
    "mls":            [0.0198,  0.0195, 0.0221, 0.0196, 0.0142],
    "_delegation_id": ["D002",  "D006", "D006", "D011", "D011"],
    "_source":        ["oracle"] * 5,
})
ledger

In [ ]:
# Two questions the provenance columns make answerable at all:
print("evaluations per delegation:")
print(ledger.groupby("_delegation_id").size().to_string())

print("\nwhich delegation produced the best feasible design?")
feasible = ledger.query("mcs >= 0.80 and mls <= 0.02")
print(feasible.loc[feasible["sigma_crit"].idxmax(), ["sigma_crit", "_delegation_id"]].to_string())

Note what the second query did: it applied the **feasibility rules** before picking a winner.
The largest `sigma_crit` in the table is not the answer — `0.7111` is, and only because it
passes both criteria.

Most of the honesty machinery in this framework exists to make that step non-optional.

# Decision 3
## A metered oracle and an append-only ledger

## The oracle is the one metered path

The **evaluator** is ground truth: the function that scores a design. It is the *only* call
counted against the evaluation budget.

Surrogates and optimizers the agents build on top are their own business, and are **not**
metered — fitting a GP a thousand times costs us nothing we care about.

Every real evaluation is written **once**, under a lock, to a shared canonical ledger, and
stamped with the delegation that produced it.

The store is protected: a stray write that would *shrink* it, or reset a completed evaluation,
is refused.

**The headline number in the final deliverable must trace back to rows in this ledger** — or
the deliverable cannot reproduce it, and the run does not close.

## Budgets: one hard cap, everything else a nudge

A long autonomous run needs guardrails, but the wrong guardrail silently truncates the science.

<br>

| kind | what | why |
| :-- | :-- | :-- |
| **soft** | the evaluation budget | *nudges* the strategizer when spending heavily — never hard-stops the science |
| **hard** | per-delegation memory | host safety. A runaway process takes down the machine, not just the run |

<br>

The asymmetry is deliberate: **a science budget must never be a hard stop.** A framework that
kills a run at eval 200 has decided a scientific question by accountancy.

Speaker notes.

There is a real story behind the soft/hard split, and it is worth telling if there is time.
A wall-clock or eval hard cap feels responsible. But in practice the agent reasons about the
cap, and once a limit is presented as a property of the world rather than a budgeting default,
it will treat the limit as a boundary on the *design space*. We watched exactly that happen —
see the lessons at the end of this lecture.

# Decision 4
## Falsification instead of a reward

## The problem with "better"

With a reference solution, progress is measurable: the error went down.

Without one, "better" is whatever the system decides to call better — and an LLM asked to
evaluate its own work is a fluent, confident, and unreliable judge of it.

So we did not give the system a reward to maximize. We gave it a **standard of proof**.

A **hypothesis** is a claim with a testable criterion, a registered prediction, and a verdict.
These live in the **hypothesis ledger**, with four statuses and no others:

`OPEN` → `SUPPORTED` / `FALSIFIED` / `INCONCLUSIVE`

And the rules for what counts as evidence live in **one file**, quoted verbatim into both the
strategizer's and the critic's instructions — so either can cite a clause and the other defers
to the same words. No paraphrase drift, no negotiating what falsification means.

## The falsification charter (abridged)

The framework injects this text verbatim. Clause numbering is stable and citable
(*"Charter §3"*).

In [ ]:
CHARTER = """
§1  A hypothesis is ONE falsifiable claim carrying a registered prediction -
    the observable whose occurrence would refute the claim.

§2  ATTEMPT and VERDICT are distinct. Before a hypothesis may be closed, an
    adequate falsification ATTEMPT must have been made - a SEVERE test: one
    that genuinely probes the registered prediction and could have refuted the
    claim had it been false (a token probe is not adequate).
    [...] For a prediction whose refutation turns on FINDING an instance,
    severity means the search had the POWER to find that instance had it
    existed. A search that merely stopped improving is an INADEQUATE test:
    failing to find a better instance is not the same as showing none exists.

§3  A hypothesis is FALSIFIED if and only if an ADEQUATE test of its registered
    prediction yields a contradiction. Concretely:
      - adequate test, prediction contradicted     -> FALSIFIED (you may not
        decline the verdict to protect a favoured claim);
      - adequate test, prediction NOT contradicted -> SUPPORTED;
      - inadequate or confounded test              -> INCONCLUSIVE. A
        contradiction from a flawed test indicts the test, not the hypothesis
        (Duhem-Quine).

§4  No moving the goalposts. A FALSIFIED verdict must rest on the contradiction
    of the SAME prediction that was registered - not a post-hoc observation
    chosen after seeing the data (the Texas-sharpshooter fallacy).

§5  SUPPORTED is corroboration, not proof. You never "confirm" a hypothesis;
    you only fail to falsify it.

§6  OPEN = no adequate test yet. The three closing statuses must cite a real
    delegation ID plus a CONCRETE result from it that bears on the registered
    prediction. What is forbidden is closing on prose or vibes.
"""
print(CHARTER)

## The three clauses that do the real work

**§2 — severity, not labels.** *"Adequacy is a property of the test's SEVERITY, never of a
label."* An agent can tag a delegation as a falsification attempt; the tag **records** the
attempt, it cannot make a weak test adequate.

**§2 again — "stopped improving" is not evidence.** This one clause kills the single most
common false conclusion in optimization:

> *"I searched and found nothing better, therefore nothing better exists."*

To close an existence claim you must argue your search **had the power to find** the thing had
it been there — by coverage, by a surrogate that predicts *the claim's own observable* above
chance, or by a theoretical bound.

**§3 — Duhem–Quine.** A contradiction from a flawed test indicts *the test*, not the
hypothesis. This is why `INCONCLUSIVE` exists as a first-class outcome, and why it is
reserved for an inadequate test rather than used as a shrug.

Speaker notes.

Worth pausing on §2's existence-claim clause, because it is the one that connects directly to
what this course already taught. "My BO run plateaued" is not evidence of a global optimum.
The charter forces the agent to state *why* its search had power — which in practice means it
must talk about coverage in low dimension, or show that its surrogate actually predicts the
quantity in the claim. Students have seen both ideas; here they become an admissibility rule.

Also worth noting: writing this charter was not a software task. It went through several
internally inconsistent drafts, and one version routed the same outcome to INCONCLUSIVE in one
clause and SUPPORTED in another. Two agents then cited the same broken text at each other. The
file now carries a warning header telling any future editor to read all six clauses before
touching one.

# Decision 5
## An adversarial critic, and a gate the run cannot talk its way through

## The deliverable *is* the work

The output of a run is a Jupyter notebook, `pipeline.ipynb`. It is **not** a summary written
afterwards:

* its markdown cells are the write-up
* its **code cells re-derive the headline result from the canonical ledger**

Before a run may close, the notebook goes through the **reproduction gate**: it is executed
end to end in a clean sandbox, and the number it produces is compared against the number the
run claims.

**A run that cannot reproduce its own headline does not pass.**

Outcome is recorded as `GATED` (passed the critic *and* reproduces) or `UNGATED`/`FAILED`
(treat the result as unaudited).

If you check one thing about a run, check this.

## Why a *reproduction* gate and not just a reviewer

An LLM critic can be argued with. A notebook that must run in a clean sandbox cannot.

<br>

The two together cover different failure modes:

| failure | caught by |
| :-- | :-- |
| the claim overstates the evidence | the **critic** |
| the number cannot be regenerated | the **gate** |
| the narrative describes work that never ran | the **critic**, reading as a skeptical peer |
| the notebook quietly hardcodes the answer | the **gate**, on a clean ledger |

<br>

Neither alone is sufficient. The gate proves a notebook *runs and reproduces*; it cannot prove
the notebook is **good science**.

Speaker notes.

The last line of that table is a real bug class we hit: a notebook can "reproduce" a number it
simply wrote down. The gate's job is narrow and mechanical; judging whether the notebook
*explains* rather than merely *reports* is the critic's, and ultimately ours. In practice we
read every deliverable as a skeptical peer on two axes: does it give a causal account of why
the result holds, and is every method it narrates one that actually ran.

# Decision 6
## Guards chosen by reversibility

## The temptation, and why it is wrong

Every time an agent does something you did not want, there is an obvious fix: **block it.**

Do that a dozen times and you have built a bureaucracy — a system that spends its budget being
refused, by a framework that has quietly substituted its own judgement for the scientist's.

So we classify every guard by **reversibility**, not by how much the behavior annoyed us:

| mode | when | behavior |
| :-- | :-- | :-- |
| **PROCEED + TIP** | easily reversible | let it through, say what looked off |
| **CONFIRM (two-shot)** | reversible but weighty | first call refuses with an explanation; an identical second call proceeds |
| **PRECONDITION-BLOCK** | impossible until the world changes | refuse, and say what would have to change |

And a rule about the message itself: every one states **what**, **why**, and **the next step**
— a helpful collaborator's tip, never a bureaucratic refusal.

In [ ]:
def two_shot_confirm():
    """The pattern, in miniature: refuse once with a reason, then honour the intent."""
    pending = set()

    def close_hypothesis(hyp_id, *, has_falsification_attempt):
        if has_falsification_attempt:
            return f"{hyp_id}: SUPPORTED (attempt on record)"
        if hyp_id not in pending:
            pending.add(hyp_id)
            return (f"{hyp_id}: refused once. Charter §5 - SUPPORTED means the claim "
                    f"survived an attempt to refute it, and none is on record. "
                    f"Re-call identically to confirm, with a written justification.")
        return f"{hyp_id}: SUPPORTED, recorded WITH the missing-attempt caveat attached"

    return close_hypothesis


close = two_shot_confirm()
print(close("H4", has_falsification_attempt=False))
print()
print(close("H4", has_falsification_attempt=False))   # identical second call

Note what the second call does **not** do: it does not pretend the attempt happened. It
proceeds *and carries the caveat forward* into the record.

The guard's job is to make a choice **visible and attributable**, not to win the argument.

Speaker notes.

The "floor" that stays hard, and why: schema validation, graph connectivity, notebook validity,
the per-delegation memory cap, reproduction-must-run, and closing-a-hypothesis-requires-evidence.
Those are provenance and host safety. Everything else was re-examined and most of it converted
to a nudge.

The clearest case we converted: a cap of three simultaneously-open hypotheses. It was closure
*discipline*, not safety — and the moment we asked the system to explore several design families
at once, three was simply wrong. It is now a two-shot confirm.

# Lessons learned
## The part that cost us the most

## Lesson 1 — Do not overfit the prompt

When a bug is "fixed" by adding a rule to an agent's prompt, that rule must pass a test first:

<br>

> Would a philosopher of science, reading this rule in isolation, nod at it as a general
> methodological principle — or frown at it as a workaround for one case?

**Parsimonious** — a philosopher nods:

> *"A hypothesis cannot be marked SUPPORTED without a falsification attempt on record."*

Popperian. Applies to every run that will ever happen.

**Overfit** — a philosopher frowns:

> *"When calling HypothesisUpdate, pass a single ID, not a comma-separated list."*

That patches one tool-call error and names no principle.

**The fallback matters as much as the test.** When a failure is real but the obvious rule is
overfit: state the underlying principle instead — or, if none can be stated, **fix it in code**
(a validation, a guard, an assertion at the tool boundary), not in the prompt.

A prompt full of special cases is a system nobody can reason about, including itself.

## Lesson 2 — "Bulletproof" has a precise meaning

We once let each design space own its own physical data store. Then every place that counts
evaluations had to remember to aggregate across stores.

It didn't — in **seven** places, each discovered one validation run at a time.

Then a fifth data point exposed an **eighth**: the design space could be chosen at the call
site independently of the delegation, so the registry-keyed fix was blind whenever the two
diverged.

The root cause was not any of the eight sites. It was that **the data model did not match the
question the whole codebase asks** — *"how many evaluations in this run?"*

Every aggregation helper was a band-aid: it made the *right* read available while leaving the
*wrong* read still present and still **looking correct**. So the next consumer was born blind.

## Lesson 2, stated as a rule

<br>

> **A mechanism is bulletproof only when the wrong usage is impossible or loud — not when the
> right usage is merely available.**

<br>

Prefer designs where the invariant the code already assumes stays **true**, over designs that
add a parallel structure every consumer must learn about.

When a parallel structure is unavoidable, enumerate the blind paths it creates and make them
loud — **before** writing code, not after the third validation run.

This one generalizes far beyond agents. It is a claim about API design, and you will meet it
again the first time you add an optional argument that callers must remember to pass.

Speaker notes.

This is the most transferable lesson in the lecture and it is worth spending time on. Ask the
room: how many of you have written a helper called `get_all_x()` alongside the existing
`get_x()`? That is the shape of the mistake. The fix is not a better helper; it is making
`get_x()` correct so there is nothing to remember.

## Lesson 3 — Self-reports are leads, not diagnoses

Every node writes a retrospective when a run closes: what contradicted itself, the most
uncertain decision, what was counterintuitive, what blocked it.

These are the **highest-signal artifact we have.** A single friction entry often names a root
cause that no traceback shows.

And they can be **wrong about mechanism.**

One run blamed a *"silent Abaqus crash"* for a 63% evaluation failure rate. The raw logs showed
it was the **license server saturating at 16-way concurrency**.

Same symptom, entirely different fix — and a fix aimed at the reported cause would have
achieved nothing while looking like diligence.

**Treat a retrospective's account of *what happened* as a lead, and verify the mechanism
against the raw logs before you trust it or act on it.**

## Lesson 4 — Never let a cost argument masquerade as physics

Our problem statement described a solve-time cap as *"a hard property"*, and said a design
family that cannot fit inside it *"is not searchable"*.

<br>

Both justifications for that cap were **cost** arguments.

The consequence: a run closed with **5.7 of its 12 hours unspent**, reasoning — correctly,
given what it had been told — that the one remaining escape from a mechanism it had just
established required a solver regime the infrastructure "cannot afford."

Presenting a budgeting default as a property of the oracle **turned an accounting choice into a
boundary on the design space.** The agent then reasoned honestly and rigorously about a
boundary that did not exist.

<br>

Six consecutive runs closing early was a symptom of this — not six independent judgement calls.

Speaker notes.

This is the single most instructive failure in the project, because nothing malfunctioned.
The agent's reasoning was sound. Its own retrospective drew exactly the right distinction:
"'My search stopped improving' would not have justified closing; 'the one remaining mechanism
requires a solver regime the infrastructure cannot afford' is a statement about the space and
the tooling, not about my search."

That is better epistemics than most of us apply. It reached a wrong conclusion because *we*
wrote a false premise into the brief. The lesson is about us, not it.

## Lesson 5 — Measuring your own system is harder than you think

A run summary should state what the run cost. It is the only number that makes *"was this run
worth it"* answerable, and runs are not uniform — **\$20.68 to \$54.45** across six runs at
comparable wall clock.

But the recorded total is **not** the cost.

The strategizer's persistent connection never reports usage, so one run logged
`total_cost_usd: None`, one call, and zero tokens — **for the node that wrote nine hypotheses
from a 1.1 MB transcript.**

Every recorded total therefore *understates* the run. Recovering the missing spend from the
transcripts: **\$54.45 recorded + \$1.58 = \$56.03 actual.**

<br>

Instrument the thing you will be asked to justify, and then **check the instrument** — a field
that exists and is populated is not the same as a field that is correct.

## Lesson 6 — A brief should not babysit

> Write the problem statement the way a PI would brief a **first-year graduate student.**

<br>

A first-year student has freedom. They also ask questions when something is unclear, and that
is normal — for a student and for an agentic workflow.

What a good brief **must** pin down:

* **objective and success criteria** — the headline number or claim
* **design space** — every variable, with bounds, type, and **units**
* **what "valid" means** — feasibility limits, regimes of validity, thresholds
* **deliverables** — anything required beyond the notebook

What it should **not** do: prescribe the method. A brief that specifies the sampler and the
surrogate has hired a technician, not a researcher — and wasted the one thing the system
uniquely brings.

## Lesson 7 — Find plumbing bugs without the model

Two kinds of test, and the order matters:

<br>

| | what it is | cost |
| :-- | :-- | :-- |
| **headless smoke** | fast, deterministic, **no live LLM** — the backend is stubbed. Exercises the plumbing | seconds |
| **end-to-end** | a full real run driving actual agents through a study | minutes to hours, real money, non-deterministic |

<br>

**Find plumbing bugs with headless tests first**, and add a regression test for the exact
failure mode. Use an end-to-end run to check *behavior* — last, and never to hunt a plumbing bug.

The temptation is always to just run it again and watch. That is the most expensive debugger
ever built, and it is non-deterministic, so it cannot even confirm your fix worked.

## What you provide, and what you get

After all of that machinery, the interface is deliberately one file:

```
my_study/
  PROBLEM_STATEMENT.md   # required: the brief the agents work from
  config.yaml            # optional: model, budgets, how a design is scored
  workspace/
    evaluator.py         # optional: your ground truth, if you ship one
```

```python
from a3dasm import AgenticRun

report = AgenticRun(study_dir="studies/my_study").execute()
```

And what comes back:

* **`pipeline.ipynb`** — the deliverable, which reproduces its own headline
* **`runs/<timestamp>/experiment_data/`** — every real evaluation, on the record
* **`runs/<timestamp>/run_status.json`** — `GATED` or not
* **`runs/<timestamp>/debug/`** — retrospectives, critic reviews, diagnostics, delegations

## Summary

* The architecture followed from the **epistemic situation**: no reference solution, expensive
  flaky evaluations, and an objective where a real number can still not count.
* **Falsification replaced a reward** because there was nothing to score against — and the
  charter lives in one citable file, quoted identically to both adjudicating agents.
* **Provenance is not optional**: every evaluation stamped, the ledger append-only and locked,
  the headline required to trace back to rows in it.
* **The gate is mechanical** — a notebook that must run in a clean sandbox cannot be argued with.
* **Guards by reversibility**, so the framework nudges rather than substitutes its judgement
  for the scientist's.
* And the lessons that cost most: don't overfit the prompt; make the wrong usage loud;
  verify self-reports against raw logs; and **never write a cost argument into the brief as
  though it were physics.**

## Next lecture

We point it at a real research problem — the **supercompressible metamaterial** from
Bessa, Glowacki & Houlder (2019) — and go through what actually happened across
**23 runs and 28 distinct design ideas.**

<br>

Including the headline it almost reported, the twelve times we changed the rules underneath it,
and the honest answer to *"did it beat the human?"*